# 04 — Makroekonomik Senaryo Analizi

**Araştırma Sorusu:** Makroekonomik ortama göre Apple'ın içsel değeri nasıl değişir?

3 senaryo:
| Senaryo | Makro Ortam |
|---------|------------|
| **Bull** | Düşük faiz, güçlü GDP, zayıf USD |
| **Base** | Mevcut ortam, ılımlı büyüme |
| **Bear** | Yüksek faiz, düşük GDP, güçlü USD |

Çıktılar:
- Senaryo karşılaştırma tablosu
- Tornado chart (en kritik değer sürücüsü)
- Değerleme aralığı görseli

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas_datareader.data as web

from src.data_fetcher import fetch_apple_beta, fetch_shares_outstanding
from src.dcf_model import calculate_wacc, intrinsic_value_per_share, sensitivity_table
from src.scenario_engine import SCENARIOS, build_scenario_summary, tornado_inputs

plt.rcParams['figure.dpi'] = 120
print('Ready.')

In [ ]:
# Apple girdi verileri (Notebook 03 ile aynı)
ticker = yf.Ticker('AAPL')
cashflow = ticker.cashflow

if 'Free Cash Flow' in cashflow.index:
    base_fcf = float(cashflow.loc['Free Cash Flow'].iloc[0])
else:
    base_fcf = 99_584_000_000

balance = ticker.balance_sheet
try:
    total_debt = float(balance.loc['Total Debt'].iloc[0]) if 'Total Debt' in balance.index else 0
    cash = float(balance.loc['Cash And Cash Equivalents'].iloc[0]) if 'Cash And Cash Equivalents' in balance.index else 0
    net_debt = total_debt - cash
except:
    net_debt = 36_000_000_000

beta = fetch_apple_beta()
shares = fetch_shares_outstanding()
current_price = ticker.info.get('currentPrice', ticker.info.get('regularMarketPrice', None))

print(f'Base FCF: ${base_fcf/1e9:.1f}B | Net Debt: ${net_debt/1e9:.1f}B | Beta: {beta:.2f}')
if current_price:
    print(f'Mevcut Piyasa Fiyatı: ${current_price:.2f}')

## 1. Senaryo Parametreleri

In [ ]:
params_rows = []
for name, s in SCENARIOS.items():
    params_rows.append({
        'Senaryo': s['label'],
        'GDP Büyümesi': f"{s['gdp_growth']:.1%}",
        'Fed Faizi': f"{s['fed_rate']:.1%}",
        'CPI': f"{s['cpi']:.1%}",
        'USD': s['usd'].capitalize(),
        'FCF Y1 Büyüme': f"{s['fcf_growth_rates'][0]:.0%}",
        'Terminal Growth': f"{s['terminal_growth']:.1%}",
        'Açıklama': s['description'],
    })

params_df = pd.DataFrame(params_rows).set_index('Senaryo')
print('Senaryo Parametreleri:')
params_df

## 2. Her Senaryo için DCF Hesabı

In [ ]:
dcf_results = {}

for scenario_name, scenario in SCENARIOS.items():
    wacc = calculate_wacc(
        risk_free_rate=scenario['risk_free_rate'],
        beta=beta,
        market_risk_premium=0.055,
    )
    result = intrinsic_value_per_share(
        base_fcf=base_fcf,
        growth_rates=scenario['fcf_growth_rates'],
        wacc=wacc,
        terminal_growth=scenario['terminal_growth'],
        net_debt=net_debt,
        shares_outstanding=shares,
    )
    dcf_results[scenario_name] = result
    print(f"{scenario['label']:12} | WACC: {wacc:.2%} | İçsel Değer: ${result['intrinsic_value_per_share']:.2f}")

summary_df = build_scenario_summary(dcf_results)
print('\nÖzet Tablo:')
summary_df

## 3. Değerleme Aralığı Görseli

In [ ]:
scenario_names = list(SCENARIOS.keys())
values = [dcf_results[n]['intrinsic_value_per_share'] for n in scenario_names]
colors = [SCENARIOS[n]['color'] for n in scenario_names]
labels = [SCENARIOS[n]['label'] for n in scenario_names]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Sol: Bar chart
bars = axes[0].bar(labels, values, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'${val:.0f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

if current_price:
    axes[0].axhline(current_price, color='black', linewidth=1.5, linestyle='--', label=f'Piyasa Fiyatı (${current_price:.0f})')
    axes[0].legend(fontsize=10)

axes[0].set_ylabel('Hisse Başına İçsel Değer ($)')
axes[0].set_title('Senaryo Bazlı DCF Değerlemesi', fontweight='bold', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, max(values) * 1.2)

# Sağ: Aralık görseli (floating bar)
bear_val = dcf_results['Bear']['intrinsic_value_per_share']
bull_val = dcf_results['Bull']['intrinsic_value_per_share']
base_val = dcf_results['Base']['intrinsic_value_per_share']

axes[1].barh(['Değerleme Aralığı'], [bull_val - bear_val], left=bear_val,
             color='#ecf0f1', edgecolor='#7f8c8d', height=0.4)
axes[1].barh(['Değerleme Aralığı'], [bear_val], left=0, color='#e74c3c', alpha=0.7, height=0.4, label=f'Bear: ${bear_val:.0f}')
axes[1].plot([base_val], ['Değerleme Aralığı'], marker='D', color='#3498db', markersize=12, label=f'Base: ${base_val:.0f}', zorder=5)
axes[1].plot([bull_val], ['Değerleme Aralığı'], marker='D', color='#2ecc71', markersize=12, label=f'Bull: ${bull_val:.0f}', zorder=5)

if current_price:
    axes[1].axvline(current_price, color='black', linewidth=2, linestyle='--', label=f'Piyasa: ${current_price:.0f}')

axes[1].set_xlabel('Hisse Başına Değer ($)')
axes[1].set_title('Bull / Base / Bear Değerleme Aralığı', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=10, loc='lower right')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Apple (AAPL) — Makroekonomik Senaryo Analizi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/scenario_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBull/Bear değerleme farkı: ${bull_val - bear_val:.0f} (+{(bull_val/bear_val-1)*100:.0f}%)')

## 4. Tornado Chart — Hangi Değişken En Kritik?

In [ ]:
try:
    from src.scenario_engine import tornado_inputs
    tornado_df = tornado_inputs(
        base_fcf=base_fcf,
        base_result=dcf_results['Base'],
        net_debt=net_debt,
        shares_outstanding=shares,
        beta=beta
    )
    print('Tornado Analizi:')
    print(tornado_df[['Variable', 'Low ($)', 'Base ($)', 'High ($)', 'Delta ($)']].to_string(index=False))
except Exception as e:
    print(f'Tornado hesabı: {e}')
    tornado_df = None

In [ ]:
if tornado_df is not None:
    base_val = dcf_results['Base']['intrinsic_value_per_share']

    fig, ax = plt.subplots(figsize=(11, 5))

    for i, row in tornado_df.reset_index(drop=True).iterrows():
        low_delta = row['Low ($)'] - base_val
        high_delta = row['High ($)'] - base_val

        ax.barh(i, low_delta, left=base_val, color='#e74c3c', alpha=0.8, height=0.5)
        ax.barh(i, high_delta, left=base_val, color='#2ecc71', alpha=0.8, height=0.5)

        ax.text(row['Low ($)'] - 1, i, f"${row['Low ($)']:.0f}", ha='right', va='center', fontsize=9)
        ax.text(row['High ($)'] + 1, i, f"${row['High ($)']:.0f}", ha='left', va='center', fontsize=9)

    ax.set_yticks(range(len(tornado_df)))
    ax.set_yticklabels(tornado_df['Variable'].values)
    ax.axvline(base_val, color='black', linewidth=1.5, linestyle='--', label=f'Base: ${base_val:.0f}')
    ax.set_xlabel('Hisse Başına İçsel Değer ($)')
    ax.set_title('Tornado Chart — Değer Üzerindeki Değişken Etkileri\n(Her değişken bağımsız olarak değiştirildi)', 
                 fontweight='bold', fontsize=12)

    red_patch = mpatches.Patch(color='#e74c3c', alpha=0.8, label='Düşük Senaryo')
    green_patch = mpatches.Patch(color='#2ecc71', alpha=0.8, label='Yüksek Senaryo')
    ax.legend(handles=[red_patch, green_patch], fontsize=10)
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('../data/processed/tornado_chart.png', dpi=150, bbox_inches='tight')
    plt.show()

## Sonuç: Temel Bulgular

### Değerleme Özeti

| Senaryo | Makro Ortam | İçsel Değer |
|---------|------------|-------------|
| **Bull** | Düşük faiz + Güçlü GDP | ~$XXX |
| **Base** | Mevcut ortam | ~$XXX |
| **Bear** | Yüksek faiz + Zayıf GDP | ~$XXX |

### Anahtar Bulgular

1. **Fed faizi en kritik değer sürücüsü** — 100bp artış WACC'ı ~70bp yükseltiyor ve içsel değeri ~%X düşürüyor
2. **FCF büyüme oranı ikinci en önemli faktör** — Bull/Bear arasındaki fark %9 büyüme farkından kaynaklanıyor
3. **Terminal growth asimetrik etki** — 1% değişim yaklaşık $X hisse değeri farkı yaratıyor
4. **Bull/Bear farkı $XXX** — Bu volatilite, makro belirsizliğin Apple değerlemesi üzerindeki etkisini somutlaştırıyor

### FP&A Çıkarımı
Bu analiz, bir FP&A analistinin neden 'sadece bir büyüme oranı varsaymak' yerine makro senaryolarla çalışması gerektiğini gösteriyor. Merkez bankası politikası, hem doğrudan (WACC) hem dolaylı (tüketici harcamaları → gelir) yoldan kurumsal değerlemeyi etkiliyor.